# node formation history

In [ ]:
import osmnx as ox
import utca
import networkx as nx
import pandas as pd
import geopandas as gpd
from tqdm import tqdm
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import matplotlib.colors as mcolors
import matplotlib.ticker as mtick

In [ ]:
import matplotlib as mpl

mpl.rcParams.update({
    # Font
    "font.size": 9,
    "axes.labelsize": 9,
    "axes.titlesize": 9,
    "legend.fontsize": 8,
    "xtick.labelsize": 8,
    "ytick.labelsize": 8,

    # Lines and markers
    "lines.linewidth": 1.2,
    "lines.markersize": 4,

    # Axes
    "axes.linewidth": 0.8,
    "xtick.major.width": 0.8,
    "ytick.major.width": 0.8,

    # Layout
    "figure.constrained_layout.use": True,
})

cm = 1 / 2.54

In [ ]:
def node_age_check(node_id: int, G: nx.Graph) -> str:
    """
    Checks if streets appeared historically at an intersection in the expected order.

    What is the expected order:
    - T: the horizontal part of the T came before the vertical stem
    - X: the newer street crossed over the older one
    - Y: a new intersection was created on an existing street

    Return values:
    - `sequential` for the above mentioned cases
    - `neutral` if all streets appeared at the same time
    - `anomaly` for all other cases, if the streets appeared in an unexpected order
    - `other` if the type is not TYX

    :param node_id: Description
    :param G: Description
    """
    node_data = G.nodes[node_id]
    edge_keys = list(node_data['angles'].keys())
    ages = [G.edges[*x].get('year') for x in edge_keys]
    if any(pd.isna(a) for a in ages):
        return 'uncertain'
    if node_data['type'] == 'T':
        angles = list(node_data['angles'].values())
        edge180_idx = np.argmax(angles)
        vertical_age = ages[edge180_idx-2]
        horiz1_age = ages[edge180_idx]
        horiz2_age = ages[edge180_idx-1]
        if (horiz1_age < vertical_age) and (horiz2_age < vertical_age):
            return 'sequential'
        elif (horiz1_age == vertical_age) and (horiz2_age == vertical_age):
            return 'neutral'
        else:
            return 'anomaly'
    elif node_data['type'] == 'X':
        if ages[0] == ages[2] and ages[1] == ages[3]:
            if ages[0] == ages[1]:
                return 'neutral'
            else:
                return 'sequential'
        else:
            return 'anomaly'
    elif node_data['type'] == 'Y':
        ages_sorted = sorted(ages)
        if ages_sorted[0] == ages_sorted[1]:
            if ages_sorted[2] == ages_sorted[1]:
                return 'neutral'
            else:
                return 'sequential'
        else:
            return 'anomaly'
    return 'other'

In [ ]:
def kerulet_node_hist(kerulet, fill_na=False, drop_uncertain=False):
    G = ox.load_graphml(f"output/bp_ker_simplified/{kerulet}.graphml")
    G = utca.prepare_graph(G)
    edges = ox.graph_to_gdfs(G, nodes=False)
    joined = utca.join_historical_streets(edges, fill_na=fill_na)
    G2 = ox.graph_from_gdfs(*utca.rebuild_neat_graph(joined))
    G2 = utca.prepare_graph(G2)
    G2.add_nodes_from(utca.apply_on_nodes(G2, node_age_check, 'edge_history'))
    nodes = ox.graph_to_gdfs(G2, edges=False)
    percentage_by_type = pd.crosstab(
        nodes["type"],
        nodes["edge_history"],
        normalize="index"
    ) * 100
    percentage_by_type.drop('other', axis=0, inplace=True)
    percentage_by_type.drop('other', axis=1, inplace=True)

    #ax = percentage_by_type.plot.barh(stacked=True, figsize=(8,4), colormap='viridis')
    #ax.set_xlabel('Percentage')
    #ax.set_ylabel('')
    #ax.legend(title='edge_history', bbox_to_anchor=(1.02, 1), loc='upper left')
    #plt.tight_layout()
    #plt.show()

    if drop_uncertain:
        ser = nodes["edge_history"][~nodes["edge_history"].isin(["uncertain", "other"])]
    else:
        ser = nodes["edge_history"]
    #edge_history_dist = nodes["edge_history"].value_counts(normalize=True) * 100
    edge_history_dist = ser.value_counts(normalize=True) * 100
    return edge_history_dist
    ax = edge_history_dist.plot.bar(stacked=True, figsize=(8, 4), colormap='cividis')
    ax.set_xlabel('edge_history')
    ax.set_ylabel('Percentage')
    plt.tight_layout()
    plt.show()
    
    return percentage_by_type.round(2)

In [ ]:
kerulet_node_hist(3, fill_na=False)

In [ ]:
results = []
for k in tqdm(range(1, 23)):
    ser = kerulet_node_hist(k, fill_na=False)
    ser.name = k
    results.append(ser)

cols = ['anomaly', 'neutral', 'sequential', 'uncertain', 'other']
all_districts_df = pd.DataFrame(results).reindex(columns=cols).fillna(0)
all_districts_df.index.name = 'kerulet'

all_districts_df

In [ ]:
observable10 = sns.color_palette([
    '#4269d0',
    '#efb118',
    '#ff725c',
    '#6cc5b0',
    '#3ca951',
    '#ff8ab7',
    '#a463f2',
    '#97bbf5',
    '#9c6b4e',
    '#9498a0',
]) #observable10

In [ ]:
palette = sns.color_palette('bright6')
palette

In [ ]:
colors = {
    'anomaly': 'red',
    'neutral': 'blue',
    'sequential': 'green',
    'uncertain': 'yellow',
    'other': 'grey'
}
palette_dict = dict(zip(cols, sns.color_palette("tab10", len(cols))))
colors = {
    'anomaly': palette[3],
    'neutral': palette[4],
    'sequential': palette[5],
    'uncertain': '0.5',
    'other': '0.3'
}
colors2 = {
    'anomaly': observable10[2],
    'neutral': observable10[1],
    'sequential': observable10[0],
    'uncertain': '0.6',
    'other': '0.5'
}
cmap = mcolors.ListedColormap(colors2.values())

In [ ]:
ax = all_districts_df.plot.barh(stacked=True, color=colors2, figsize=(5*cm,20*cm), legend=False)
ax.set_ylabel('District')
sns.despine(ax=ax, trim=True, left=True, bottom=True, right=True, top=True)
#ax.set_xticklabels([])
#axs[0].set_xticks([])
ax.xaxis.set_major_formatter(mtick.PercentFormatter(xmax=100.0))
#ax.set_title('Districts')
#ax.legend(title='edge_history', bbox_to_anchor=(0.5, -0.15), loc='upper center', ncol=5)
#plt.tight_layout()

In [ ]:
def plot_all_district_bar(save=False):
    results = []
    for k in tqdm(range(1, 23)):
        ser = kerulet_node_hist(k, fill_na=False)
        ser.name = k
        results.append(ser)
    
    cols = ['anomaly', 'neutral', 'sequential', 'uncertain', 'other']
    all_districts_df = pd.DataFrame(results).reindex(columns=cols).fillna(0)
    all_districts_df.index.name = 'kerulet'
    
    all_districts_df

    fig, ax = plt.subplots(figsize=(5*cm,20*cm))
    all_districts_df.plot.barh(stacked=True, color=colors2, legend=False, ax=ax)
    ax.set_ylabel('District')
    sns.despine(ax=ax, trim=True, left=True, bottom=True, right=True, top=True)
    #ax.set_xticklabels([])
    #axs[0].set_xticks([])
    ax.xaxis.set_major_formatter(mtick.PercentFormatter(xmax=100.0))
    #ax.set_title('Districts')
    #ax.legend(title='edge_history', bbox_to_anchor=(0.5, -0.15), loc='upper center', ncol=5)
    #plt.tight_layout()
    if save:
        fig.savefig('output/figs_maj10/node_hist_all_distr.pdf', bbox_inches='tight')
    return fig

In [ ]:
plot_all_district_bar()

In [ ]:
def plot_kerulet_node_hist(kerulet, fill_na=False, drop_uncertain=False):
    G = ox.load_graphml(f"output/bp_ker_simplified/{kerulet}.graphml")
    G = utca.prepare_graph(G)
    edges = ox.graph_to_gdfs(G, nodes=False)
    joined = utca.join_historical_streets(edges, fill_na=fill_na)
    G2 = ox.graph_from_gdfs(*utca.rebuild_neat_graph(joined))
    G2 = utca.prepare_graph(G2)
    G2.add_nodes_from(utca.apply_on_nodes(G2, node_age_check, "edge_history"))
    nodes, edges = ox.graph_to_gdfs(G2)

    fig, ax = plt.subplots(figsize=(10*cm, 6*cm))
    #nodes.plot(
    #    ax=ax,
    #    column="edge_history",
    #    legend=True,
    #    cmap=cmap,
    #    markersize=0.1,
    #    categories=cols,
    #    legend_kwds={"title": "Type"},
    #    #style_kwds={"marker": "o"},
    #)
    #ax.scatter(data=nodes, x='x', y='y', c='edge_history', cmap=cmap)
    edges.plot(ax=ax, color='black', alpha=0.5, linewidth=0.2)
    sns.scatterplot(data=nodes, x='x', y='y', hue='edge_history', palette=colors2, ax=ax, s=3, legend=False)
    ax.set_axis_off()
    return fig

In [ ]:
fig12 = plot_kerulet_node_hist(12)

In [ ]:
#fig12.savefig('output/figs_maj10/node_hist_test3.pdf', bbox_inches='tight')

In [ ]:
plot_kerulet_node_hist(19)

# tyx distribution within districts

In [ ]:
cmap2 = mcolors.ListedColormap(list(colors2.values())[:-1])

In [ ]:
def plot_percentage_type(df):
    fig, ax = plt.subplots(figsize=(8*cm,4*cm))
    df.plot.barh(stacked=True, colormap=cmap2, legend=False, ax=ax)
    #ax.set_xlabel('Percentage')
    ax.set_ylabel('')
    sns.despine(ax=ax, trim=True, left=True, bottom=True, right=True, top=True)
    #ax.set_xticklabels([])
    ax.set_xticks([])
    #ax.set_axis_off()
    #ax.legend(title='edge_history', bbox_to_anchor=(1.02, 1), loc='upper left')
    plt.tight_layout()
    plt.show()

In [ ]:
def kerulet_node_hist_bar(kerulet, fill_na=False):
    G = ox.load_graphml(f"output/bp_ker_simplified/{kerulet}.graphml")
    G = utca.prepare_graph(G)
    edges = ox.graph_to_gdfs(G, nodes=False)
    joined = utca.join_historical_streets(edges, fill_na=fill_na)
    G2 = ox.graph_from_gdfs(*utca.rebuild_neat_graph(joined))
    G2 = utca.prepare_graph(G2)
    G2.add_nodes_from(utca.apply_on_nodes(G2, node_age_check, "edge_history"))
    nodes = ox.graph_to_gdfs(G2, edges=False)
    percentage_by_type = pd.crosstab(
        nodes["type"],
        nodes["edge_history"],
        normalize="index"
    ) * 100
    percentage_by_type.drop('other', axis=0, inplace=True)
    percentage_by_type.drop('other', axis=1, inplace=True)

    plot_percentage_type(percentage_by_type)

In [ ]:
kerulet_node_hist_bar(19)

In [ ]:


def kerulet_kombinalt_plot(kerulet):
    fig, axs = plt.subplots(2,1, figsize=(8*cm, 10*cm), height_ratios=[1,3])
    G = ox.load_graphml(f"output/bp_ker_simplified/{kerulet}.graphml")
    G = utca.prepare_graph(G)
    edges = ox.graph_to_gdfs(G, nodes=False)
    joined = utca.join_historical_streets(edges, fill_na=False)
    G2 = ox.graph_from_gdfs(*utca.rebuild_neat_graph(joined))
    G2 = utca.prepare_graph(G2)
    G2.add_nodes_from(utca.apply_on_nodes(G2, node_age_check, "edge_history"))
    nodes, edges = ox.graph_to_gdfs(G2)
    
    percentage_by_type = pd.crosstab(
        nodes["type"],
        nodes["edge_history"],
        normalize="index"
    ) * 100
    percentage_by_type.drop('other', axis=0, inplace=True)
    percentage_by_type.drop('other', axis=1, inplace=True)
    percentage_by_type.plot.barh(stacked=True, colormap=cmap2, legend=False, ax=axs[0])
    #ax.set_xlabel('Percentage')
    axs[0].set_ylabel('')
    sns.despine(ax=axs[0], trim=True, left=True, bottom=True, right=True, top=True)
    #ax.set_xticklabels([])
    #axs[0].set_xticks([])
    axs[0].set_title(f'District {kerulet}')
    axs[0].xaxis.set_major_formatter(mtick.PercentFormatter(xmax=100.0))

    edges.plot(ax=axs[1], color='black', alpha=0.5, linewidth=0.2)
    sns.scatterplot(data=nodes, x='x', y='y', hue='edge_history', palette=colors2, ax=axs[1], s=7, legend=False, alpha=1)
    axs[1].set_axis_off()
    return fig


In [ ]:
ker19 = kerulet_kombinalt_plot(19)

In [ ]:
ker12 = kerulet_kombinalt_plot(12)

In [ ]:
#ker19.savefig('output/figs_maj10/node_hist_19.pdf', bbox_inches='tight')
#dpi=300

#ker12.savefig('output/figs_maj10/node_hist_12.pdf', bbox_inches='tight')

In [ ]:
#ker19.savefig('output/figs_maj10/node_hist_19.png', dpi=300)
#ker12.savefig('output/figs_maj10/node_hist_12.png', dpi=300)

In [ ]:
from matplotlib.patches import Patch

def legend_from_colors2(colors_map, order=None, figsize=(3*cm, 2*cm)):
    if order is None:
        order = list(colors_map.keys())
    handles = [Patch(facecolor=colors_map[k], label=k) for k in order]
    fig, ax = plt.subplots(figsize=figsize)
    ax.axis('off')
    ax.legend(handles=handles, loc='center', frameon=True, ncol=5, title='Node history')
    return fig

legend_fig = legend_from_colors2(colors2, order=cols, figsize=(1*cm, len(cols)*0.7*cm))
legend_fig

In [ ]:
#legend_fig.savefig('output/figs_maj10/node_hist_legend.pdf', bbox_inches='tight')